[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nagomi-tech/blog-aibeginner/blob/main/Colab/english_speech_emotion.ipynb)


# 音声感情分析：英語編（VAD理論の実演 + OpenAI TTS実験）

英語学習済みの音声感情認識モデル `audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim` を使い、
OpenAI TTS（`gpt-4o-mini-tts`）で生成した感情音声から、Valence（感情価）・Arousal（覚醒度）・
Dominance（支配性）の3次元スコアを推論します。

## このノートブックの内容
1. 環境セットアップ
2. VAD理論の簡単な実演（象限イメージ）
3. audeeringモデルのロード（重み手動抽出版）
4. 推論関数の定義
5. OpenAI TTSで感情音声を生成（実験1：異なる文面＋対応感情）
6. OpenAI TTSで感情音声を生成（実験2：同一文面＋instructionsのみ変更）
7. 推論の実行とVAD空間の可視化
8. 結果のCSV保存

> 日本語編（AivisSpeech音声＋JVNVでのファインチューニング）は別ノートブックで扱います。


## 1. 環境セットアップ

In [ ]:
!pip install -q transformers huggingface_hub openai soundfile librosa matplotlib pandas japanize-matplotlib


In [ ]:
import os
import io
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語フォントを自動設定（グラフの日本語表示対応）
import soundfile as sf
import librosa

from transformers import Wav2Vec2Processor, AutoConfig
from transformers.models.wav2vec2.modeling_wav2vec2 import Wav2Vec2Model
from huggingface_hub import snapshot_download

os.makedirs("audio_samples", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("セットアップ完了")


## 2. VAD理論の簡単な実演

Valence（横軸：快〜不快）・Arousal（縦軸：落ち着き〜興奮）の2軸に、代表的な感情語を
プロットしてみます（値はあくまで概念的な目安です）。Dominance軸は色の濃淡で表現しています。


In [ ]:
# 概念的なVAD値の例（あくまでイメージ。厳密な学術値ではありません）
emotion_examples = {
    "happy":       (0.85, 0.60, 0.70),
    "excited":     (0.75, 0.85, 0.65),
    "content":     (0.75, 0.20, 0.55),
    "angry":       (0.15, 0.80, 0.70),
    "fear":        (0.15, 0.75, 0.20),
    "sad":         (0.20, 0.25, 0.25),
    "calm":        (0.65, 0.15, 0.55),
    "neutral":     (0.50, 0.50, 0.50),
}

fig, ax = plt.subplots(figsize=(7, 7))
for name, (v, a, d) in emotion_examples.items():
    ax.scatter(v, a, s=300 * (0.3 + d), c=[d], cmap="viridis", vmin=0, vmax=1, edgecolors="black")
    ax.annotate(name, (v, a), textcoords="offset points", xytext=(8, 8))

ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Valence (negative -> positive)")
ax.set_ylabel("Arousal (calm -> excited)")
ax.set_title("VAD空間のイメージ（色の濃さ = Dominance）")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("results/vad_theory_illustration.png", dpi=150)
plt.show()


## 3. audeeringモデルのロード

`audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim` は独自のモデルクラスを使います。

**注意**：このモデルのconfig上の`classifier_proj_size`は256と記載されていますが、
実際に保存されている重みファイル（分類ヘッドの`dense`層）のサイズは1024です。
config記載と実際の重みファイルが食い違っているため、`from_pretrained`を素直に使うと
サイズ不一致エラーになります。

そのため、①モデルの構造は1024に固定して自前で定義し、②重み（state_dict）だけを
チェックポイントファイルから手動で抽出してロードする、という方式を取ります。


In [ ]:
class RegressionHead(nn.Module):
    """VAD(Valence, Arousal, Dominance)を出力する回帰ヘッド

    注: config.classifier_proj_size(256)ではなく、実際のチェックポイントの
    サイズ(1024)に合わせて固定値を設定している。
    """

    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, 1024)
        self.dropout = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(1024, config.num_labels)

    def forward(self, features, **kwargs):
        x = features
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class EmotionModel(nn.Module):
    """音声感情分類器（VAD回帰）

    Wav2Vec2PreTrainedModelを継承せず、nn.Moduleを直接継承することで
    from_pretrained周りの内部処理（重みタイイング等）の互換性問題を回避している。
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = RegressionHead(config)

    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)
        hidden_states = outputs[0]
        hidden_states = torch.mean(hidden_states, dim=1)
        logits = self.classifier(hidden_states)
        return hidden_states, logits


MODEL_NAME = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用デバイス: {device}")

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)
model = EmotionModel(config).to(device)

# チェックポイントをダウンロードし、重みを手動で読み込む
model_cache_dir = snapshot_download(MODEL_NAME)
state_dict_path = os.path.join(model_cache_dir, "pytorch_model.bin")
checkpoint = torch.load(state_dict_path, map_location="cpu")

# wav2vec2本体の重み
wav2vec2_state_dict = {
    k.replace("wav2vec2.", "", 1): v
    for k, v in checkpoint.items()
    if k.startswith("wav2vec2.")
}
missing, unexpected = model.wav2vec2.load_state_dict(wav2vec2_state_dict, strict=False)
print("wav2vec2 missing keys:", missing)
print("wav2vec2 unexpected keys:", unexpected)

# 分類ヘッドの重み
classifier_state_dict = {
    k.replace("classifier.", "", 1): v
    for k, v in checkpoint.items()
    if k.startswith("classifier.")
}
model.classifier.load_state_dict(classifier_state_dict, strict=True)

model.eval()
print("モデルのロードが完了しました。")


## 4. 推論関数の定義

In [ ]:
TARGET_SR = 16000


def load_audio(path, target_sr=TARGET_SR):
    speech, sr = librosa.load(path, sr=target_sr)
    return speech


@torch.no_grad()
def predict_vad(path, model=model, processor=processor, device=device):
    """音声ファイルパスを受け取り、[arousal, dominance, valence]を返す

    公式ドキュメント準拠：出力ロジットの順序は [arousal, dominance, valence]
    """
    speech = load_audio(path)
    inputs = processor(speech, sampling_rate=TARGET_SR, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    _, logits = model(input_values)
    arousal, dominance, valence = logits.squeeze().cpu().numpy().tolist()

    return {"arousal": arousal, "dominance": dominance, "valence": valence}


print("推論関数の準備ができました。predict_vad(path) で実行できます。")


## 5. OpenAI TTSで感情音声を生成

Colabの「シークレット」機能（左サイドバーの鍵アイコン）に `OPENAI_API_KEY` を登録してください。

### 実験1：異なる文面＋対応する感情指示
5種類の異なる文面に、それぞれ対応する感情語を含めたうえで、`instructions`パラメータでも
同じ感情を指示します。文面自体に感情語が含まれるため、「声の抑揚」と「言葉の意味」の
どちらが効いているかは切り分けられない点に注意してください。


In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

TTS_MODEL = "gpt-4o-mini-tts"
VOICE = "alloy"


def synthesize(text, instructions, out_path, voice=VOICE):
    with client.audio.speech.with_streaming_response.create(
        model=TTS_MODEL,
        voice=voice,
        input=text,
        instructions=instructions,
    ) as response:
        response.stream_to_file(out_path)
    return out_path


# 実験1：異なる文面 + 対応する感情語を含む指示
experiment1_items = [
    {
        "emotion": "happy",
        "text": "I just got the best news of my life, I can't stop smiling!",
        "instructions": "Speak in a joyful, delighted, upbeat tone as if genuinely happy.",
    },
    {
        "emotion": "sad",
        "text": "I feel so down today, everything just seems heavy and gray.",
        "instructions": "Speak in a sad, low-energy, heavy-hearted tone.",
    },
    {
        "emotion": "angry",
        "text": "This is completely unacceptable, I am furious about how this was handled.",
        "instructions": "Speak in an angry, sharp, frustrated tone.",
    },
    {
        "emotion": "fear",
        "text": "I'm really scared, something feels terribly wrong right now.",
        "instructions": "Speak in a fearful, anxious, trembling tone.",
    },
    {
        "emotion": "neutral",
        "text": "The report was submitted on time and reviewed by the team yesterday.",
        "instructions": "Speak in a flat, neutral, matter-of-fact tone.",
    },
]

exp1_paths = []
for item in experiment1_items:
    out_path = f"audio_samples/exp1_{item['emotion']}_en.wav"
    synthesize(item["text"], item["instructions"], out_path)
    exp1_paths.append({"emotion": item["emotion"], "path": out_path})
    print(f"生成完了: {out_path}")


### 実験2：完全に同一の文面＋instructionsだけを変える

文面をすべて同一（感情語を含まない業務連絡文）にし、`instructions`だけを変えることで、
「同じ言葉を、指示された感情でどれだけ演じ分けられているか」を純粋に検証します。


In [ ]:
NEUTRAL_TEXT = (
    "Tomorrow's meeting has been postponed due to the department manager's schedule. "
    "Once a new date is decided, we will contact you by email."
)

experiment2_instructions = {
    "neutral": "Speak in a flat, neutral, matter-of-fact tone.",
    "happy":   "Speak in a joyful, delighted, upbeat tone as if genuinely happy.",
    "sad":     "Speak in a sad, low-energy, heavy-hearted tone.",
    "angry":   "Speak in an angry, sharp, frustrated tone.",
    "fear":    "Speak in a fearful, anxious, trembling tone.",
}

exp2_paths = []
for emotion, instructions in experiment2_instructions.items():
    out_path = f"audio_samples/exp2_{emotion}_01_en.wav"
    synthesize(NEUTRAL_TEXT, instructions, out_path)
    exp2_paths.append({"emotion": emotion, "path": out_path})
    print(f"生成完了: {out_path}")


## 6. 推論の実行

In [ ]:
def run_experiment(paths_info, experiment_name):
    rows = []
    for item in paths_info:
        vad = predict_vad(item["path"])
        rows.append({
            "experiment": experiment_name,
            "emotion_label": item["emotion"],
            "path": item["path"],
            **vad,
        })
    return pd.DataFrame(rows)


df_exp1 = run_experiment(exp1_paths, "exp1_different_text")
df_exp2 = run_experiment(exp2_paths, "exp2_same_text")

df_all = pd.concat([df_exp1, df_exp2], ignore_index=True)
df_all


## 7. VAD空間の可視化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (exp_name, title) in zip(
    axes,
    [("exp1_different_text", "実験1：異なる文面＋対応感情"),
     ("exp2_same_text", "実験2：同一文面＋instructionsのみ変更")],
):
    sub = df_all[df_all["experiment"] == exp_name]
    sc = ax.scatter(sub["valence"], sub["arousal"], c=sub["dominance"],
                     cmap="viridis", vmin=0, vmax=1, s=200, edgecolors="black")
    for _, row in sub.iterrows():
        ax.annotate(row["emotion_label"], (row["valence"], row["arousal"]),
                    textcoords="offset points", xytext=(8, 8))
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Valence")
    ax.set_ylabel("Arousal")
    ax.set_title(title)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.colorbar(sc, ax=ax, label="Dominance")

plt.tight_layout()
plt.savefig("results/vad_english_experiments.png", dpi=150)
plt.show()


## 8. 結果のCSV保存

In [ ]:
csv_path = "results/vad_english_results.csv"
df_all.to_csv(csv_path, index=False)
print(f"結果を保存しました: {csv_path}")

df_all


## まとめ

- Arousal・Dominanceは、実験1・実験2ともに直感と一致する妥当な傾向（angryで高Arousal、
  angryで高Dominance／fearで低Dominanceなど）が確認できました。
- Valenceは実験1（異なる文面）では明確な傾向が出ましたが、これは文面自体に感情語が
  含まれていたためである可能性が残ります。実験2（同一文面）でも一定の違いは出たものの、
  実験1ほど明確ではありませんでした。
- この結果は、audeeringモデルの元論文（Wagner et al., 2023）が指摘する
  「Valenceの精度は暗黙的に学習した言語情報に支えられている」という点と整合的です。

次のノートブック（日本語編）では、同じモデルをファインチューニングなしで日本語音声に
適用し、Valence・Arousal・Dominanceそれぞれがどう変化するかを検証します。
